# Cross-Situational Tone Learning Simulation

This notebook runs the simulation code for the CSL tone-sandhi model directly in Google Colab. It clones the GitHub repository, runs the core models, and displays the production-demo outputs.

The generated audio is a sonification of model-selected F0 contours, not natural speech synthesis.


In [ ]:
# Setup: clone the repository into Colab and install dependencies.
# This cell is safe to re-run.
import os
from pathlib import Path

REPO_URL = "https://github.com/xlphon/csl-tone-simulation.git"
REPO_DIR = Path("/content/csl-tone-simulation")
WORK_DIR = REPO_DIR / "csl-tone-simulation-v2"

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {WORK_DIR}
!pip -q install -r requirements.txt


## 1. Import modules

In [ ]:
import glob
import os
import pandas as pd
from IPython.display import Audio, Image, display

from csl_tone_model import run_sim, param_sweep_tau
from productive_sandhi_with_citation import ToneCSLModelV3b
from csl_tone_model import build_exposure_trials


## 2. Base model: word memory and classificatory generalisation

This quick run uses fewer simulations than the thesis analysis so that it finishes quickly in Colab.


In [ ]:
mem, gen = run_sim(n_sims=100, tau=7.0, noise=11.0, seed=42)
print(f"Word memory:       mean={mem.mean():.3f}, sd={mem.std():.3f}")
print(f"Generalisation:    mean={gen.mean():.3f}, sd={gen.std():.3f}")


## 3. Tau sweep

In [ ]:
tau_values = [0.5, 2.0, 5.0, 7.0, 10.0, 20.0]
sweep = param_sweep_tau(tau_values, n_sims=50, noise=11.0, seed=42)
pd.DataFrame({
    "tau": sweep["tau"],
    "word_memory_mean": sweep["mem_m"],
    "word_memory_sd": sweep["mem_s"],
    "generalisation_mean": sweep["gen_m"],
    "generalisation_sd": sweep["gen_s"],
})


## 4. Productive sandhi with citation mappings

In [ ]:
model = ToneCSLModelV3b(tau=7.0, noise=11.0)
model.train(build_exposure_trials(), seed=4)
result = model.test_productive_sandhi(verbose=False)
print(f"Productive sandhi, both syllables correct: {result['both']:.3f}")
print(f"Sigma 1 correct: {result['sigma1']:.3f}")
print(f"Sigma 2 correct: {result['sigma2']:.3f}")


## 5. Render production demo outputs

This creates CSV tables, F0 plots, and audio sonifications. The segment layer is evaluated separately in `segment_memory_outputs.csv`.


In [ ]:
!python render_production_demo.py --tau 7.0 --noise 11.0 --seed 4 --n-examples 6 --output-dir demo_outputs


## 6. Inspect the productive-tone table

In [ ]:
production = pd.read_csv("demo_outputs/production_outputs.csv")
production[[
    "construction", "word1", "word2", "target_segments", "citation",
    "target_tones", "predicted_tones", "correct_sigma1", "correct_sigma2",
    "correct_tones_both", "segment_eval_note"
]].head(12)


## 7. Inspect the segment-memory diagnostic

In [ ]:
segment_memory = pd.read_csv("demo_outputs/segment_memory_outputs.csv")
segment_memory.head(12)


## 8. Display one F0 plot and one audio sonification

In [ ]:
plot_files = sorted(glob.glob("demo_outputs/f0_plots/*.png"))
audio_files = sorted(glob.glob("demo_outputs/audio/*_predicted.wav"))

if plot_files:
    display(Image(filename=plot_files[0]))
if audio_files:
    display(Audio(audio_files[0]))


## 9. Optional: strict noun-only citation cue

This version records only the overt noun citation cue, which is a stricter interpretation of the behavioural exposure phase.


In [ ]:
!python render_production_demo.py --strict-noun-only --tau 7.0 --noise 11.0 --seed 4 --n-examples 6 --output-dir demo_outputs_noun_only
pd.read_csv("demo_outputs_noun_only/production_outputs.csv").head(12)
